# Physics-Anchored Relative-State (PARS) Model
### Reliability-weighted relative-heading states + partial-amplitude correction

This notebook is the compact, reproducible entry point for the current **PARS** model.

The model deliberately keeps the absolute level simple:

$$
B0_i = C - \theta_{i}^{*}
$$

and permits time variation only when label-free SCADA evidence supports a persistent state change:

$$
\hat y_i(t) = B0_i - \beta\,\Delta r_i(t)$$.

Here, \($\theta_i^{*}$\) is the long-term median of rolling power-vs-vane argmax estimates, while
\($\Delta r_i(t)$\) is a sparse, reliability-weighted relative-heading state correction.

### Current validation result

Reference local run on the three labelled PPP turbines:

| Holdout | Model MAE | Constant MAE | Model RMSE | Constant RMSE |
|---|---:|---:|---:|---:|
| PPP_WTG12 | 0.357 | 0.357 | 0.608 | 0.608 |
| PPP_WTG13 | 0.608 | 1.004 | 0.979 | 1.387 |
| PPP_WTG14 | 0.206 | 0.206 | 0.262 | 0.262 |
| **Macro** | **0.390** | **0.522** | **0.616** | **0.752** |

The important behaviour is structural: WTG12 and WTG14 collapse to the constant prior, while WTG13 receives two persistent state corrections.

This notebook does **not** write a submission CSV.

## 1. Reproducibility and data split

The validation split is strict leave-one-turbine-out (LOTO):

- labelled development turbines: `PPP_WTG12`, `PPP_WTG13`, `PPP_WTG14`;
- unlabeled targets inspected after model fitting: `PPP_WTG17`, `SSS_WTG06`;
- all published turbines may contribute **unlabeled** same-site context to the state detector.

The target/holdout labels are never used to construct their SCADA states.

In [1]:
from pathlib import Path
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# Find repository root without a machine-specific absolute path.
HERE = Path.cwd().resolve()
ROOT = HERE

if not (ROOT / "src").exists():
    matches = [
        parent
        for parent in [HERE, *HERE.parents]
        if (parent / "src").exists()
    ]
    if not matches:
        raise FileNotFoundError(
            "Could not locate repository root containing src/."
        )
    ROOT = matches[0]

zip_candidates = [
    ROOT / "turbines_data.zip",
    ROOT.parent / "turbines_data.zip",
]
ZIP_PATH = next(
    (path for path in zip_candidates if path.exists()),
    None,
)

if ZIP_PATH is None:
    raise FileNotFoundError(
        "Place turbines_data.zip in the repository root or its parent directory."
    )

sys.path.insert(0, str(ROOT / "src"))

from baseline_loto_ridge import read_turbine, list_turbines, wrap_180
from fleet_context import (
    FleetConfig,
    load_layout,
    bin_turbine,
    circular_median_deg,
)
from yaw_relative_state import RelativeStateConfig
from fleet_context import circular_mean_deg
from yaw_model import (
    ModelConfig,
    estimate_theta_star_map,
    build_turbine_bundle,
    apply_site_common_mode_veto,
    fit_global_beta,
    predict_from_bundle,
    score_prediction,
)

TRAIN = ["PPP_WTG12", "PPP_WTG13", "PPP_WTG14"]
TARGETS = ["PPP_WTG17", "SSS_WTG06"]

FLEET = FleetConfig()
RELATIVE = RelativeStateConfig()
MODEL = ModelConfig()

print("Repository:", ROOT)
print("SCADA archive:", ZIP_PATH)

Repository: E:\EnergyHacks\github_release
SCADA archive: E:\EnergyHacks\turbines_data.zip


## 2. Physics anchor: long-term turbine reference

The constant prior is retained as the default prediction because it has very low variance.

For each turbine, the frozen B0 estimator computes a rolling apparent power-optimal vane angle
\($\hat\theta_{\text{argmax}}(t)$\). The long-term turbine-specific reference is

$$
\theta_i^{*} = \operatorname{median}_t \hat\theta_{\text{argmax},i}(t).
$$

The labelled training turbines calibrate one shared absolute offset:

$$
C = \operatorname{mean}_{i\in\text{train}} \left(\bar y_i + \theta_i^{*}\right).
$$

The deployed default is therefore

$$
B0_i=C-\theta_i^{*}.
$$

Dynamic state logic is not allowed to replace this absolute anchor; it can only make sparse corrections around it.

In [2]:
t0 = time.perf_counter()

turbines = list_turbines(str(ZIP_PATH))
raw = {
    turbine: read_turbine(str(ZIP_PATH), turbine)
    for turbine in turbines
}

layout = load_layout(ROOT, turbines)
binned = {
    turbine: bin_turbine(raw[turbine], FLEET)
    for turbine in turbines
}

labels = {
    turbine: (
        raw[turbine]
        .assign(date=pd.to_datetime(raw[turbine]["date"]))
        .groupby("date")["yaw_misalignment_deg"]
        .median()
        .sort_index()
    )
    for turbine in TRAIN
}

theta_star = estimate_theta_star_map(
    raw,
    TRAIN + TARGETS,
)

LOAD_SECONDS = time.perf_counter() - t0

theta_table = (
    pd.Series(theta_star, name="theta_star_deg")
    .rename_axis("turbine")
    .to_frame()
)

display(theta_table.round(3))
print(f"Load + B0 estimation: {LOAD_SECONDS:.1f}s")

,theta_star_deg
turbine,
PPP_WTG12,-3.444
PPP_WTG13,0.352
PPP_WTG14,-4.354
PPP_WTG17,-2.483
SSS_WTG06,-1.513


Load + B0 estimation: 174.3s


## 3. Label-free relative-heading state detector

The dynamic channel uses only SCADA and same-site turbine context.

For target \(i\) and neighbour \(j\):

1. form circular heading differences;
2. remove pair/sector baselines;
3. estimate pair reliability from coverage, residual MAD and distance;
4. robustly aggregate usable pairs into a daily target-relative heading residual;
5. construct a neighbour-only background residual to identify common-mode motion.

Two complementary change detectors are used on the cleaned relative signal:

- a persistent rolling before/after detector;
- a conservative L2 state segmentation used only for long two-sided regimes.

A candidate is rejected when it is better explained by:

- sensor/encoder-like jumps;
- the configured sensor-shadow window;
- local background/common-mode motion;
- insufficient pair agreement;
- same-site synchronous events;
- weak event confidence.

Only surviving high-confidence events are allowed to move the prediction away from the constant prior.

In [3]:
t0 = time.perf_counter()

# Build every published turbine so the site-common-mode veto uses the full
# unlabeled site context rather than only the labelled turbines.
all_bundles = {
    turbine: build_turbine_bundle(
        turbine,
        binned,
        layout,
        FLEET,
        RELATIVE,
        MODEL,
    )
    for turbine in turbines
}

all_bundles = apply_site_common_mode_veto(
    all_bundles,
    MODEL,
)

bundles = {
    turbine: all_bundles[turbine]
    for turbine in TRAIN + TARGETS
}

MODEL_BUILD_SECONDS = time.perf_counter() - t0

# Pair reliability summary.
pair_rows = []
for turbine, bundle in bundles.items():
    table = bundle["pair_diagnostics"].reset_index(names="neighbour")
    table.insert(0, "turbine", turbine)
    pair_rows.append(
        table[
            [
                "turbine",
                "neighbour",
                "coverage",
                "pair_mad",
                "reliability_weight",
                "usable",
            ]
        ]
    )

pair_diagnostics = pd.concat(pair_rows, ignore_index=True)
display(pair_diagnostics.round(3))

# Keep the full event table in memory, but show only accepted events and
# important vetoes in the notebook.
event_rows = []
for turbine, bundle in bundles.items():
    table = bundle["boundaries"]
    if table is None or table.empty:
        continue

    out = table.reset_index(names="date")
    out.insert(0, "turbine", turbine)
    event_rows.append(out)

event_diagnostics = (
    pd.concat(event_rows, ignore_index=True)
    if event_rows
    else pd.DataFrame()
)

if not event_diagnostics.empty:
    important_reasons = {
        "yaw",
        "pelt_yaw",
        "sensor",
        "sensor_shadow",
        "site_common_mode",
        "low_confidence_prior",
    }

    event_view = event_diagnostics[
        event_diagnostics["accepted"]
        | event_diagnostics["reason"].isin(important_reasons)
    ][
        [
            "turbine",
            "date",
            "source",
            "change",
            "z",
            "pair_agreement",
            "event_confidence",
            "shrinkage",
            "reason",
            "accepted",
        ]
    ].copy()

    numeric = event_view.select_dtypes(include=[np.number]).columns
    event_view[numeric] = event_view[numeric].round(3)
    display(event_view.sort_values(["turbine", "date"]))

print(f"Relative-heading model build: {MODEL_BUILD_SECONDS:.1f}s")

,turbine,neighbour,coverage,pair_mad,reliability_weight,usable
0,PPP_WTG12,PPP_WTG11,0.625,3.555,0.256,True
1,PPP_WTG12,PPP_WTG13,0.653,6.055,0.129,True
2,PPP_WTG12,PPP_WTG14,0.614,8.402,0.043,True
3,PPP_WTG13,PPP_WTG14,0.817,5.033,0.275,True
4,PPP_WTG13,PPP_WTG12,0.707,6.192,0.124,True
5,PPP_WTG13,PPP_WTG11,0.832,7.804,0.058,True
6,PPP_WTG13,PPP_WTG08,0.843,7.669,0.039,True
7,PPP_WTG14,PPP_WTG13,0.774,5.210,0.253,True
8,PPP_WTG14,PPP_WTG12,0.608,8.662,0.035,True
9,PPP_WTG14,PPP_WTG08,0.768,6.125,0.059,True


,turbine,date,source,change,z,pair_agreement,event_confidence,shrinkage,reason,accepted
6,PPP_WTG12,2023-07-09,pelt,-33.198,23.921,0.000,0.000,0.000,sensor,False
7,PPP_WTG12,2023-07-17,rolling,-32.451,34.241,0.750,0.263,0.000,sensor,False
8,PPP_WTG12,2023-08-07,rolling,-8.654,8.388,0.900,0.000,0.000,sensor_shadow,False
9,PPP_WTG12,2023-08-08,pelt,-8.396,4.798,0.900,0.000,0.000,sensor_shadow,False
13,PPP_WTG12,2024-03-11,rolling,-6.030,3.093,0.699,0.000,0.000,site_common_mode,False
16,PPP_WTG13,2023-02-19,rolling,-3.614,7.227,0.751,0.321,0.321,yaw,True
20,PPP_WTG13,2023-07-27,pelt,-3.603,2.236,0.884,0.436,0.436,pelt_yaw,True
23,PPP_WTG13,2024-03-10,rolling,-6.261,5.067,0.671,0.000,0.000,site_common_mode,False
31,PPP_WTG14,2023-06-05,rolling,-3.435,3.259,0.680,0.142,0.000,low_confidence_prior,False
34,PPP_WTG14,2024-03-11,rolling,6.584,3.226,0.839,0.000,0.000,site_common_mode,False


Relative-heading model build: 27.5s


## 4. Strict turbine-level LOTO validation

Each fold:

1. holds out one labelled turbine;
2. estimates \(C\) and any eligible global \($\beta$\) using only the other labelled turbines;
3. uses the holdout turbine only through its unlabeled SCADA-derived prior/state features;
4. compares the state-aware prediction with the same-fold constant prior.

Because only a few reliable labelled state transitions survive, the model deliberately falls back to the physical prior \($\beta=1$\) instead of fitting an unstable slope.

The notebook reports MAE/RMSE. Legacy micro-step boundary metrics returned by the helper are intentionally not used for model selection.

In [4]:
t0 = time.perf_counter()
rows = []

for holdout in TRAIN:
    train_ids = [
        turbine
        for turbine in TRAIN
        if turbine != holdout
    ]

    C, beta = fit_global_beta(
        train_ids,
        bundles,
        labels,
        theta_star,
        MODEL,
    )

    prediction = predict_from_bundle(
        holdout,
        bundles[holdout],
        C,
        beta,
        theta_star,
    )

    metrics = score_prediction(
        prediction,
        labels[holdout],
        C - theta_star[holdout],
    )

    boundary_table = bundles[holdout]["boundaries"]
    accepted = (
        boundary_table.loc[
            boundary_table["accepted"].fillna(False)
        ]
        if len(boundary_table)
        else pd.DataFrame()
    )

    rows.append(
        {
            "holdout": holdout,
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "constant_mae": metrics["constant_mae"],
            "constant_rmse": metrics["constant_rmse"],
            "beta": beta,
            "n_states": int(
                bundles[holdout]["observables"]["cluster"].nunique()
            ),
            "accepted_boundaries": ", ".join(
                pd.Timestamp(date).date().isoformat()
                for date in accepted.index
            )
            or "none",
        }
    )

LOTO_SECONDS = time.perf_counter() - t0
loto = pd.DataFrame(rows)

display(loto.round(3))

macro = pd.DataFrame(
    {
        "state_aware": [
            loto["mae"].mean(),
            loto["rmse"].mean(),
        ],
        "constant_prior": [
            loto["constant_mae"].mean(),
            loto["constant_rmse"].mean(),
        ],
    },
    index=["MAE", "RMSE"],
)

display(macro.round(3))

mae_gain = 1.0 - macro.loc["MAE", "state_aware"] / macro.loc["MAE", "constant_prior"]
rmse_gain = 1.0 - macro.loc["RMSE", "state_aware"] / macro.loc["RMSE", "constant_prior"]

print(
    f"Macro improvement: MAE {mae_gain:.1%}, RMSE {rmse_gain:.1%}"
)
print(f"LOTO scoring: {LOTO_SECONDS:.2f}s")

,holdout,mae,rmse,constant_mae,constant_rmse,beta,n_states,accepted_boundaries
0,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none


,state_aware,constant_prior
MAE,0.390,0.522
RMSE,0.616,0.752


Macro improvement: MAE 25.3%, RMSE 18.1%
LOTO scoring: 0.06s


## 5. PARS component audit: frozen-boundary soft pair-quality weighting

This is a state-level robustness test, not a new detector. The accepted boundaries, \(B0\), \(\beta\), and calibration protocol are unchanged. Each state's daily relative-heading observations are aggregated with a continuous weight based on cross-pair spread, finite-pair coverage, and fixed historical pair reliability. No day is hard-deleted.

In [5]:
from yaw_model import apply_pair_quality_state_weights

if any(name not in globals() for name in ("bundles", "labels", "theta_star")):
    try:
        get_ipython().run_line_magic(
            "store",
            "-r bundles labels theta_star MODEL_CONFIG",
        )
    except Exception as exc:
        raise RuntimeError(
            "Run the data/build cells first, or restore the stored model state."
        ) from exc
if "MODEL" not in globals():
    MODEL = globals().get("MODEL_CONFIG")
if MODEL is None:
    raise RuntimeError("MODEL / MODEL_CONFIG is not available.")

SOFT_SPREAD_SCALE_DEG = 2.5
soft_bundles = apply_pair_quality_state_weights(
    bundles,
    MODEL,
    spread_scale_deg=SOFT_SPREAD_SCALE_DEG,
)

from yaw_model import run_loto

stage_tables = {}
stage_macros = {}
for stage_name, stage_bundles in {
    "all_days": bundles,
    "soft_pair_quality": soft_bundles,
}.items():
    table, macro_stage = run_loto(
        TRAIN, stage_bundles, labels, theta_star, MODEL
    )
    table.insert(0, "stage", stage_name)
    stage_tables[stage_name] = table
    stage_macros[stage_name] = macro_stage

soft_loto = pd.concat(stage_tables.values(), ignore_index=True)
display(soft_loto.round(3))
display(
    soft_loto.groupby("stage")[["mae", "rmse", "constant_mae", "constant_rmse"]]
    .mean().round(3)
)

quality_rows = []
for tid, bundle in soft_bundles.items():
    q = bundle["observables"]["state_quality_weight"]
    spread = bundle["observables"]["state_pair_spread_deg"]
    quality_rows.append({
        "turbine": tid,
        "median_day_quality": q.replace(0.0, np.nan).median(),
        "median_pair_spread_deg": spread.median(),
        "usable_quality_days": int((q > 0).sum()),
    })
display(pd.DataFrame(quality_rows).round(3))
print("Boundaries and calibration are frozen; only state-level day weights changed.")

,stage,holdout,mae,rmse,constant_mae,constant_rmse,beta,states,boundaries
0,all_days,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,all_days,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,all_days,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
3,soft_pair_quality,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
4,soft_pair_quality,PPP_WTG13,0.578,0.958,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
5,soft_pair_quality,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none


,mae,rmse,constant_mae,constant_rmse
stage,,,,
all_days,0.39,0.616,0.522,0.752
soft_pair_quality,0.38,0.609,0.522,0.752


,turbine,median_day_quality,median_pair_spread_deg,usable_quality_days
0,PPP_WTG12,0.741,1.084,569
1,PPP_WTG13,0.617,1.743,716
2,PPP_WTG14,0.618,1.748,662
3,PPP_WTG17,0.794,0.744,710
4,SSS_WTG06,0.864,0.904,607


Boundaries and calibration are frozen; only state-level day weights changed.


## 6. Pair-quality weighting robustness and attribution

The scale is audited at three pre-declared values. This is not a parameter search: boundaries remain frozen, and every scale uses the same turbine-level LOTO protocol. Pair residual levels have an arbitrary offset, so the WTG13 attribution compares only duration-centred dynamic shape (with the heading sign reversed), not raw residual levels against absolute yaw labels. Target rows are sensitivity only.

In [6]:
from fleet_context import circular_mean_deg
from yaw_model import (
    apply_pair_quality_state_weights,
    weighted_circular_median_deg,
)

if any(name not in globals() for name in ("bundles", "labels", "theta_star")):
    try:
        get_ipython().run_line_magic(
            "store",
            "-r bundles labels theta_star MODEL_CONFIG",
        )
    except Exception as exc:
        raise RuntimeError(
            "Run the data/build cells first, or restore the stored model state."
        ) from exc
if "MODEL" not in globals():
    MODEL = globals().get("MODEL_CONFIG")
if MODEL is None:
    raise RuntimeError("MODEL / MODEL_CONFIG is not available.")
if "soft_bundles" not in globals():
    soft_bundles = apply_pair_quality_state_weights(
        bundles,
        MODEL,
        spread_scale_deg=2.5,
    )

WEIGHTING_SCALES = {
    "conservative": 4.0,
    "current": 2.5,
    "aggressive": 1.5,
}
robust_tables = {}
robust_bundles = {}
robust_rows = []

baseline_table, _ = run_loto(
    TRAIN, bundles, labels, theta_star, MODEL
)
baseline_table.insert(0, "weighting", "all_days")
baseline_table.insert(1, "spread_scale_deg", np.nan)
robust_rows.append(baseline_table)

for weighting_name, scale in WEIGHTING_SCALES.items():
    stage = apply_pair_quality_state_weights(
        bundles,
        MODEL,
        spread_scale_deg=scale,
    )
    robust_bundles[weighting_name] = stage
    table, _ = run_loto(TRAIN, stage, labels, theta_star, MODEL)
    table.insert(0, "weighting", weighting_name)
    table.insert(1, "spread_scale_deg", scale)
    robust_tables[weighting_name] = table
    robust_rows.append(table)

robust_loto = pd.concat(robust_rows, ignore_index=True)
display(robust_loto.round(3))
display(
    robust_loto.groupby("weighting")[["mae", "rmse", "constant_mae", "constant_rmse"]]
    .mean().round(3)
)

def state_level_attribution(bundle, turbine, label=None):
    obs = bundle[turbine]["observables"]
    rows = []
    label_daily = None
    if label is not None:
        label_daily = label.groupby(label.index.normalize()).median()
    for state, part in obs.groupby("cluster", observed=True):
        valid = part["relative_heading_smooth"].notna()
        values = part.loc[valid, "relative_heading_smooth"].to_numpy(float)
        weights = part.loc[valid, "state_quality_weight"].to_numpy(float)
        level_all = circular_median_deg(values)
        level_soft = weighted_circular_median_deg(values, weights)
        label_level = np.nan
        if label_daily is not None:
            label_level = label_daily.reindex(part.index).dropna().median()
        rows.append({
            "turbine": turbine,
            "state": int(state),
            "days": int(valid.sum()),
            "median_pair_spread_deg": part["state_pair_spread_deg"].median(),
            "effective_days": (weights.sum() ** 2 / (weights ** 2).sum()) if (weights > 0).any() else 0.0,
            "relative_level_all_deg": level_all,
            "relative_level_soft_deg": level_soft,
            "soft_minus_all_deg": wrap_180(level_soft - level_all),
            "label_level_deg": label_level,
        })
    table = pd.DataFrame(rows)
    if table.empty:
        return table

    # The relative-heading residual has an arbitrary pair-reference offset.
    # Compare only its dynamic shape with the labelled yaw states: negate the
    # heading residual and centre both trajectories with duration weights.
    duration = table["days"].to_numpy(float)
    duration = duration / duration.sum()
    yaw_all = -table["relative_level_all_deg"].to_numpy(float)
    yaw_soft = -table["relative_level_soft_deg"].to_numpy(float)
    all_centre = circular_mean_deg(yaw_all, duration)
    soft_centre = circular_mean_deg(yaw_soft, duration)
    table["heading_yaw_all_centered_deg"] = [wrap_180(x - all_centre) for x in yaw_all]
    table["heading_yaw_soft_centered_deg"] = [wrap_180(x - soft_centre) for x in yaw_soft]
    labels_arr = table["label_level_deg"].to_numpy(float)
    label_ok = np.isfinite(labels_arr)
    if label_ok.any():
        label_w = duration[label_ok] / duration[label_ok].sum()
        label_centre = circular_mean_deg(labels_arr[label_ok], label_w)
        centred_labels = np.full(len(table), np.nan)
        centred_labels[label_ok] = [wrap_180(x - label_centre) for x in labels_arr[label_ok]]
        table["label_centered_deg"] = centred_labels
        table["all_shape_error_deg"] = [
            wrap_180(a - b) if np.isfinite(b) else np.nan
            for a, b in zip(table["heading_yaw_all_centered_deg"], centred_labels)
        ]
        table["soft_shape_error_deg"] = [
            wrap_180(a - b) if np.isfinite(b) else np.nan
            for a, b in zip(table["heading_yaw_soft_centered_deg"], centred_labels)
        ]
    return table

wtg13_attr = state_level_attribution(
    robust_bundles["current"],
    "PPP_WTG13",
    labels["PPP_WTG13"],
)
print("WTG13 state-level attribution: current soft weighting")
display(wtg13_attr.round(3))

C_ref, beta_ref = fit_global_beta(
    TRAIN, bundles, labels, theta_star, MODEL
)
target_rows = []
for turbine in TARGETS:
    for stage_name, stage_bundle in {
        "all_days": bundles,
        "soft_current": robust_bundles["current"],
    }.items():
        pred = predict_from_bundle(
            turbine, stage_bundle[turbine], C_ref, beta_ref, theta_star
        )
        state_values = pred.groupby("cluster")["prediction"].median()
        target_rows.append({
            "turbine": turbine,
            "stage": stage_name,
            "beta": beta_ref,
            "states": int(pred["cluster"].nunique()),
            "state_levels_deg": ", ".join(f"{v:.3f}" for v in state_values),
            "duration_mean_deg": pred["prediction"].mean(),
            "prediction_min": pred["prediction"].min(),
            "prediction_max": pred["prediction"].max(),
        })

print("Blind-target sensitivity: same C and beta, all-days versus current soft weighting")
display(pd.DataFrame(target_rows).round(3))
print("Boundaries, C, beta, and lambda were not re-tuned in this audit.")

,weighting,spread_scale_deg,holdout,mae,rmse,constant_mae,constant_rmse,beta,states,boundaries
0,all_days,NaN,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,all_days,NaN,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,all_days,NaN,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
3,conservative,4.0,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
4,conservative,4.0,PPP_WTG13,0.595,0.968,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
5,conservative,4.0,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
6,current,2.5,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
7,current,2.5,PPP_WTG13,0.578,0.958,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
8,current,2.5,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
9,aggressive,1.5,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none


,mae,rmse,constant_mae,constant_rmse
weighting,,,,
aggressive,0.384,0.610,0.522,0.752
all_days,0.390,0.616,0.522,0.752
conservative,0.386,0.613,0.522,0.752
current,0.380,0.609,0.522,0.752


WTG13 state-level attribution: current soft weighting


,turbine,state,days,median_pair_spread_deg,effective_days,relative_level_all_deg,relative_level_soft_deg,soft_minus_all_deg,label_level_deg,heading_yaw_all_centered_deg,heading_yaw_soft_centered_deg,label_centered_deg,all_shape_error_deg,soft_shape_error_deg
0,PPP_WTG13,0,49,3.972,31.170,3.970,4.111,0.141,-9.666,-3.930,-3.986,-3.034,-0.896,-0.952
1,PPP_WTG13,1,158,2.619,119.953,1.707,2.019,0.312,-8.030,-1.667,-1.894,-1.398,-0.269,-0.496
2,PPP_WTG13,2,524,1.491,445.782,-0.830,-0.819,0.011,-5.927,0.870,0.944,0.705,0.165,0.238


Blind-target sensitivity: same C and beta, all-days versus current soft weighting


,turbine,stage,beta,states,state_levels_deg,duration_mean_deg,prediction_min,prediction_max
0,PPP_WTG17,all_days,1.0,4,"-5.246, -5.732, -6.680, -1.422",-3.602,-6.680,-1.422
1,PPP_WTG17,soft_current,1.0,4,"-5.280, -5.652, -6.635, -1.441",-3.602,-6.635,-1.441
2,SSS_WTG06,all_days,1.0,4,"-4.079, -8.162, -3.826, -3.729",-4.572,-8.162,-3.729
3,SSS_WTG06,soft_current,1.0,4,"-4.090, -7.923, -3.859, -3.910",-4.572,-7.923,-3.859


Boundaries, C, beta, and lambda were not re-tuned in this audit.


## 7. Interaction audit: state weighting × amplitude

This small audit keeps the accepted boundaries and $B0=C-\theta^\star$ fixed. It compares all-days versus soft pair-quality state levels at $\lambda\in\{0,0.75,1\}$ with fixed $\beta=1$. The three values mean stable amplitude, partial amplitude, and full amplitude; this is an interaction check, not a new parameter search.

In [12]:
from yaw_model import (
    apply_full_amplitude_stable_states,
    blend_stable_amplitude,
)

if "soft_bundles" not in globals():
    soft_bundles = apply_pair_quality_state_weights(
        bundles, MODEL, spread_scale_deg=2.5
    )

full_bundles = apply_full_amplitude_stable_states(bundles, MODEL)
soft_full_bundles = apply_full_amplitude_stable_states(soft_bundles, MODEL)

def run_fixed_beta_loto(stage_bundles):
    rows = []
    for holdout in TRAIN:
        fit_ids = [tid for tid in TRAIN if tid != holdout]
        C_fixed = float(np.mean([
            labels[tid].mean() + theta_star[tid]
            for tid in fit_ids
        ]))
        pred = predict_from_bundle(
            holdout, stage_bundles[holdout], C_fixed, 1.0, theta_star
        )
        metrics = score_prediction(
            pred, labels[holdout], C_fixed - theta_star[holdout]
        )
        boundaries = stage_bundles[holdout]["boundaries"]
        active_col = (
            "prediction_active"
            if "prediction_active" in boundaries.columns
            else "accepted"
        )
        active = boundaries[active_col].fillna(False) if len(boundaries) else pd.Series(dtype=bool)
        accepted = boundaries.index[active] if len(boundaries) else []
        rows.append({
            "holdout": holdout,
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "constant_mae": metrics["constant_mae"],
            "constant_rmse": metrics["constant_rmse"],
            "states": metrics["n_states"],
            "boundaries": ", ".join(
                pd.Timestamp(d).date().isoformat() for d in accepted
            ) or "none",
        })
    return pd.DataFrame(rows)

interaction_rows = []
for estimator, stable_stage, full_stage in (
    ("all_days", bundles, full_bundles),
    ("soft_2.5deg", soft_bundles, soft_full_bundles),
):
    for amplitude_lambda in (0.0, 0.75, 1.0):
        mixed_stage = blend_stable_amplitude(
            stable_stage, full_stage, amplitude_lambda
        )
        table = run_fixed_beta_loto(mixed_stage)
        table.insert(0, "estimator", estimator)
        table.insert(1, "lambda_amp", amplitude_lambda)
        interaction_rows.append(table)

interaction_loto = pd.concat(interaction_rows, ignore_index=True)
print("Interaction audit: fixed boundaries, B0, and beta=1")
display(interaction_loto.round(3))
display(
    interaction_loto.groupby(["estimator", "lambda_amp"])[
        ["mae", "rmse", "constant_mae", "constant_rmse"]
    ].mean().round(3)
)

Interaction audit: fixed boundaries, B0, and beta=1


,estimator,lambda_amp,holdout,mae,rmse,constant_mae,constant_rmse,states,boundaries
0,all_days,0.00,PPP_WTG12,0.357,0.608,0.357,0.608,1,none
1,all_days,0.00,PPP_WTG13,0.608,0.979,1.004,1.387,3,"2023-02-19, 2023-07-27"
2,all_days,0.00,PPP_WTG14,0.206,0.262,0.206,0.262,1,none
3,all_days,0.75,PPP_WTG12,0.357,0.608,0.357,0.608,1,none
4,all_days,0.75,PPP_WTG13,0.509,0.702,1.004,1.387,3,"2023-02-19, 2023-07-27"
5,all_days,0.75,PPP_WTG14,0.206,0.262,0.206,0.262,1,none
6,all_days,1.00,PPP_WTG12,0.357,0.608,0.357,0.608,1,none
7,all_days,1.00,PPP_WTG13,0.584,0.755,1.004,1.387,3,"2023-02-19, 2023-07-27"
8,all_days,1.00,PPP_WTG14,0.206,0.262,0.206,0.262,1,none
9,soft_2.5deg,0.00,PPP_WTG12,0.357,0.608,0.357,0.608,1,none


mae   rmse  constant_mae  constant_rmse
estimator   lambda_amp                                           
all_days    0.00        0.390  0.616         0.522          0.752
            0.75        0.357  0.524         0.522          0.752
            1.00        0.382  0.541         0.522          0.752
soft_2.5deg 0.00        0.380  0.609         0.522          0.752
            0.75        0.358  0.524         0.522          0.752
            1.00        0.382  0.541         0.522          0.752

## 8. Optional validation export

This cell prepares the selected validation candidate without writing a file by default. Set $\texttt{EXPORT\_PATH}$ explicitly before running it. The default candidate is soft pair-quality state levels with $\lambda=0.75$; change only $\texttt{EXPORT\_ESTIMATOR}$ to $\texttt{all\_days}$ for the tied all-days ablation.

In [14]:
from yaw_model import (
    apply_full_amplitude_stable_states,
    blend_stable_amplitude,
)

EXPORT_ESTIMATOR = "soft_2.5deg"  # or "all_days"
EXPORT_LAMBDA = 0.75
EXPORT_TARGET = "PPP_WTG17"
EXPORT_PATH = ROOT / "submissions" / "Results_33_T3_4.csv"
#None  # set e.g. ROOT / "submissions" / "Results_33_T3_validation_soft075.csv"

if EXPORT_ESTIMATOR == "soft_2.5deg":
    stable_stage = soft_bundles
else:
    stable_stage = bundles
full_stage = apply_full_amplitude_stable_states(stable_stage, MODEL)
candidate_stage = blend_stable_amplitude(
    stable_stage, full_stage, EXPORT_LAMBDA
)
C_export = float(np.mean([
    labels[tid].mean() + theta_star[tid] for tid in TRAIN
]))
pred_export = predict_from_bundle(
    EXPORT_TARGET, candidate_stage[EXPORT_TARGET], C_export, 1.0, theta_star
)
dates_export = pd.date_range("2023-01-01", "2024-12-31", freq="D")
out_export = pd.DataFrame(index=dates_export)
out_export["yaw_misalignment_deg"] = pred_export["prediction"].reindex(dates_export)
out_export["cluster"] = pred_export["cluster"].reindex(dates_export)
out_export["yaw_misalignment_deg"] = out_export["yaw_misalignment_deg"].fillna(C_export - theta_star[EXPORT_TARGET])
out_export["cluster"] = out_export["cluster"].fillna(0).astype(int)
out_export = out_export.reset_index(names="date")
out_export["turbine_id"] = EXPORT_TARGET
out_export["date"] = out_export["date"].dt.strftime("%Y-%m-%d")
out_export = out_export[["turbine_id", "date", "yaw_misalignment_deg", "cluster"]]
print(f"candidate={EXPORT_ESTIMATOR}, lambda={EXPORT_LAMBDA}, beta=1, C={C_export:.6f}")
print(f"rows={len(out_export)}, range=({out_export.yaw_misalignment_deg.min():.3f}, {out_export.yaw_misalignment_deg.max():.3f})")
if EXPORT_PATH is None:
    print("No CSV written. Set EXPORT_PATH explicitly to export.")
else:
    EXPORT_PATH = Path(EXPORT_PATH)
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    out_export.to_csv(EXPORT_PATH, index=False, float_format="%.6f")
    print(f"Wrote {len(out_export)} rows to {EXPORT_PATH}")

candidate=soft_2.5deg, lambda=0.75, beta=1, C=-6.085367
rows=731, range=(-7.945, 0.261)
Wrote 731 rows to E:\EnergyHacks\github_release\submissions\Results_33_T3_4.csv


## 9. PARS final fit and unlabeled target diagnostics

The final calibration uses all three labelled training turbines.

Target diagnostics below are **not validation scores**. They are shown to make deployment behaviour auditable:

- accepted state dates;
- detector source and confidence;
- correction range around the constant prior;
- resulting prediction range.

This is especially important because the targets may contain larger unlabeled state structure than the training turbines.

In [7]:
FINAL_C, FINAL_BETA = fit_global_beta(
    TRAIN,
    bundles,
    labels,
    theta_star,
    MODEL,
)

predictions = {
    turbine: predict_from_bundle(
        turbine,
        bundles[turbine],
        FINAL_C,
        FINAL_BETA,
        theta_star,
    )
    for turbine in TRAIN + TARGETS
}

diagnostic_rows = []

for turbine in TRAIN + TARGETS:
    bundle = bundles[turbine]
    boundary_table = bundle["boundaries"]

    accepted = (
        boundary_table.loc[
            boundary_table["accepted"].fillna(False)
        ]
        if len(boundary_table)
        else pd.DataFrame()
    )

    correction = bundle["observables"][
        "relative_prior_correction"
    ].fillna(0.0)

    pred = predictions[turbine]["prediction"]

    diagnostic_rows.append(
        {
            "turbine": turbine,
            "constant_prior": FINAL_C - theta_star[turbine],
            "accepted_boundaries": ", ".join(
                pd.Timestamp(date).date().isoformat()
                for date in accepted.index
            )
            or "none",
            "accepted_sources": ", ".join(
                accepted["source"].astype(str).tolist()
            )
            if len(accepted)
            else "none",
            "mean_event_confidence": (
                float(accepted["event_confidence"].mean())
                if len(accepted)
                else 0.0
            ),
            "correction_range": float(
                correction.max() - correction.min()
            ),
            "prediction_min": float(pred.min()),
            "prediction_max": float(pred.max()),
        }
    )

deployment_diagnostics = pd.DataFrame(diagnostic_rows)

display(deployment_diagnostics.round(3))

print(f"Final C = {FINAL_C:.3f}")
print(f"Final beta = {FINAL_BETA:.3f}")
print(
    {
        "load_and_B0_seconds": round(LOAD_SECONDS, 2),
        "relative_model_seconds": round(MODEL_BUILD_SECONDS, 2),
        "LOTO_seconds": round(LOTO_SECONDS, 2),
    }
)

# Basic release sanity checks.
for turbine, pred in predictions.items():
    values = pred["prediction"].to_numpy(dtype=float)
    assert np.isfinite(values).all(), f"Non-finite prediction for {turbine}"
    assert np.max(np.abs(values)) <= 90.0, f"Prediction out of bounds for {turbine}"

print("Release sanity checks passed.")

,turbine,constant_prior,accepted_boundaries,accepted_sources,mean_event_confidence,correction_range,prediction_min,prediction_max
0,PPP_WTG12,-2.642,none,none,0.000,0.000,-2.642,-2.642
1,PPP_WTG13,-6.437,"2023-02-19, 2023-07-27","rolling, pelt",0.379,1.631,-7.764,-6.133
2,PPP_WTG14,-1.731,none,none,0.000,0.000,-1.731,-1.731
3,PPP_WTG17,-3.602,"2023-06-25, 2023-10-15, 2024-01-07","rolling, rolling, rolling",0.512,5.258,-6.680,-1.422
4,SSS_WTG06,-4.572,"2023-05-28, 2023-09-24, 2024-10-13","rolling, rolling, rolling",0.542,4.434,-8.162,-3.729


Final C = -6.085
Final beta = 1.000
{'load_and_B0_seconds': 174.32, 'relative_model_seconds': 27.47, 'LOTO_seconds': 0.06}
Release sanity checks passed.


## 10. Interpretation and limitations

PARS is intentionally conservative.

**What the validation supports**

- The long-term B0 prior is already strong for turbines whose yaw level is effectively static.
- Persistent relative-heading states add value when there is real temporal structure, as seen on WTG13.
- Sensor/reference events must be excluded across all detector sources; otherwise a state detector can turn an encoder event into a false yaw correction.
- The available labels do not support learning a complex amplitude mapping, so \($\beta=1$\) remains the preferred physical prior when transition evidence is sparse.

**What is not established**

- Good LOTO performance on three PPP turbines does not prove the magnitude of unlabeled PPP17/SSS06 state corrections.
- SSS06 is a cross-site transfer and remains the highest domain-shift risk.
- The model should therefore be treated as a constant-prior model with sparse corrections, not as a general daily yaw estimator.

## 11. Validation results visualization

In [8]:
# ============================================================
# Blind validation / final-target state summaries
# ============================================================

def prediction_state_summary(turbine, predictions):
    pred = predictions[turbine].copy()

    pred["date"] = pd.to_datetime(pred.index)
    pred["cluster"] = pred["cluster"].astype(int)

    summary = (
        pred.groupby("cluster")
        .agg(
            start=("date", "min"),
            end=("date", "max"),
            days=("date", "size"),
            yaw_prediction=("prediction", "median"),
            constant_prior=("constant_prior", "median"),
            relative_correction=(
                "relative_prior_correction",
                "median",
            ),
        )
        .reset_index()
    )

    return summary


print("PPP_WTG17 — blind validation prediction")
display(
    prediction_state_summary(
        "PPP_WTG17",
        predictions,
    ).round(3)
)

print("SSS_WTG06 — blind final-target prediction")
display(
    prediction_state_summary(
        "SSS_WTG06",
        predictions,
    ).round(3)
)

PPP_WTG17 — blind validation prediction


C:\Users\HJL\AppData\Local\Temp\ipykernel_2768\418222005.py:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,cluster,start,end,days,yaw_prediction,constant_prior,relative_correction
0,0,2023-01-01,2023-06-24,175,-5.246,-3.602,1.644
1,1,2023-06-25,2023-10-14,112,-5.732,-3.602,2.130
2,2,2023-10-15,2024-01-06,84,-6.680,-3.602,3.078
3,3,2024-01-07,2024-12-31,360,-1.422,-3.602,-2.180


SSS_WTG06 — blind final-target prediction


C:\Users\HJL\AppData\Local\Temp\ipykernel_2768\418222005.py:43: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,cluster,start,end,days,yaw_prediction,constant_prior,relative_correction
0,0,2023-01-01,2023-05-27,147,-4.079,-4.572,-0.494
1,1,2023-05-28,2023-09-23,119,-8.162,-4.572,3.590
2,2,2023-09-24,2024-10-12,385,-3.826,-4.572,-0.746
3,3,2024-10-13,2024-12-31,80,-3.729,-4.572,-0.843


In [9]:
validation_preview = (
    predictions["PPP_WTG17"]
    [["cluster", "prediction"]]
    .rename(
        columns={
            "prediction": "yaw_misalignment_deg",
        }
    )
    .copy()
)

validation_preview.insert(
    0,
    "date",
    validation_preview.index.strftime("%Y-%m-%d"),
)

validation_preview.insert(
    0,
    "turbine_id",
    "PPP_WTG17",
)

display(validation_preview.head())
display(validation_preview.tail())

assert len(validation_preview) == 731
assert validation_preview[
    "yaw_misalignment_deg"
].notna().all()

,turbine_id,date,cluster,yaw_misalignment_deg
2023-01-01,PPP_WTG17,2023-01-01,0,-5.245909
2023-01-02,PPP_WTG17,2023-01-02,0,-5.245909
2023-01-03,PPP_WTG17,2023-01-03,0,-5.245909
2023-01-04,PPP_WTG17,2023-01-04,0,-5.245909
2023-01-05,PPP_WTG17,2023-01-05,0,-5.245909


,turbine_id,date,cluster,yaw_misalignment_deg
2024-12-27,PPP_WTG17,2024-12-27,3,-1.422174
2024-12-28,PPP_WTG17,2024-12-28,3,-1.422174
2024-12-29,PPP_WTG17,2024-12-29,3,-1.422174
2024-12-30,PPP_WTG17,2024-12-30,3,-1.422174
2024-12-31,PPP_WTG17,2024-12-31,3,-1.422174


In [10]:
MODEL_CONFIG = globals().get("MODEL", globals().get("model_config"))

if MODEL_CONFIG is None:
    raise RuntimeError("MODEL / model_config not found")

%store bundles
%store MODEL_CONFIG

print("Stored PARS state")

Stored 'bundles' (dict)
Stored 'MODEL_CONFIG' (ModelConfig)
Stored PARS state


In [11]:
%store bundles
%store MODEL_CONFIG
%store labels
%store theta_star

print("Stored PARS state")

if "bundles" not in globals():
    raise RuntimeError("bundles not restored")

if "MODEL_CONFIG" not in globals():
    raise RuntimeError("MODEL_CONFIG not restored")

print("Repo:", ROOT)
print("Loaded PARS bundles:", list(bundles))
print("MODEL_CONFIG:", MODEL_CONFIG)

Stored 'bundles' (dict)
Stored 'MODEL_CONFIG' (ModelConfig)
Stored 'labels' (dict)
Stored 'theta_star' (dict)
Stored PARS state
Repo: E:\EnergyHacks\github_release
Loaded PARS bundles: ['PPP_WTG12', 'PPP_WTG13', 'PPP_WTG14', 'PPP_WTG17', 'SSS_WTG06']
MODEL_CONFIG: ModelConfig(beta_prior=1.0, beta_ridge=12.0, beta_min=0.8, beta_max=1.2, min_beta_changes=3, beta_window_days=21, min_event_confidence=0.25, max_event_shrinkage=0.9, site_common_mode_window_days=7, site_common_mode_min_turbines=3)
